# 04_extract_text

Dry-run text extraction on the latest inventory output. This notebook stays conservative: text-like files, PDF text extraction, DOCX paragraphs/tables, and XLSX sheet previews. No OCR yet.


In [1]:
from pathlib import Path
from datetime import datetime
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.inventory import ensure_inventory_schema
from src.extractors import ExtractConfig, enrich_inventory_with_text, save_text_outputs

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'outputs'
inventory_files = [p for p in OUTPUT_DIR.glob('inventory_*.parquet') if not p.name.startswith('inventory_with_text_')]
assert inventory_files, 'No inventory parquet files found. Run 02_inventory.ipynb first.'
# Pick the newest file by filesystem timestamp, not filename order.
INVENTORY_PATH = max(inventory_files, key=lambda p: p.stat().st_mtime)
print('Using inventory file:', INVENTORY_PATH.name)


Using inventory file: inventory_BCH01p800-01_vipe_SERRES_20260330_083028.parquet


In [2]:
inv = pd.read_parquet(INVENTORY_PATH)
inv = ensure_inventory_schema(inv)
print('Rows:', len(inv))
preview_cols = [c for c in ['relative_path', 'suffix', 'size_bytes'] if c in inv.columns]
display(inv[preview_cols].head(10))
if 'suffix' in inv.columns:
    display(inv['suffix'].fillna('').value_counts().rename_axis('suffix').reset_index(name='count').head(20))
else:
    print('suffix column unavailable after schema backfill')


Rows: 249


,relative_path,suffix,size_bytes
0,00_ASSET_MASTER\01_IDENTITY_REGISTERS\BOC0150-...,.zip,2371989
1,00_ASSET_MASTER\01_IDENTITY_REGISTERS\ΓΕΜΗ_AET...,.pdf,85343
2,00_ASSET_MASTER\01_IDENTITY_REGISTERS\ΓΕΜΗ_AET...,.pdf,224444
3,00_ASSET_MASTER\01_IDENTITY_REGISTERS\ΓΕΜΗ_AET...,.pdf,135040
4,00_ASSET_MASTER\01_IDENTITY_REGISTERS\ΓΕΜΗ_AET...,.pdf,134546
5,00_ASSET_MASTER\02_MASTER_DATA\01 - Τι είναι τ...,.pdf,1145979
6,00_ASSET_MASTER\02_MASTER_DATA\01 - Τι είναι τ...,.docx,85968
7,00_ASSET_MASTER\02_MASTER_DATA\01 - Τι είναι τ...,.pdf,879563
8,00_ASSET_MASTER\02_MASTER_DATA\01 - Τι είναι τ...,.pdf,10715471
9,00_ASSET_MASTER\02_MASTER_DATA\01 - Τι είναι τ...,.pdf,1090575


,suffix,count
0,.pdf,173
1,.xlsx,19
2,.msg,17
3,.docx,16
4,.pptx,6
5,.odt,4
6,,3
7,.zip,2
8,.mp4,2
9,.ods,2


In [3]:
config = ExtractConfig(
    max_chars_per_file=12000,
    max_csv_rows=30,
    max_csv_columns=20,
    max_xlsx_rows_per_sheet=30,
    max_xlsx_columns=20,
    max_docx_paragraphs=300,
    max_pdf_pages=30,
    preview_chars=300,
)
config


ExtractConfig(max_chars_per_file=12000, max_csv_rows=30, max_csv_columns=20, max_xlsx_rows_per_sheet=30, max_xlsx_columns=20, max_docx_paragraphs=300, max_pdf_pages=30, preview_chars=300)

In [4]:
enriched = enrich_inventory_with_text(inv, path_column='absolute_path', config=config)
display(enriched[['relative_path', 'suffix', 'text_status', 'text_source', 'extracted_chars', 'text_preview']].head(20))


Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 24 0 (offset 0)
Ignoring wrong pointing object 27 0 (offset 0)
Ignoring wrong pointing object 33 0 (offset 0)
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 24 0 (offset 0)
Ignoring wrong pointing object 27 0 (offset 0)
Ignoring wrong pointing object 33 0 (offset 0)
c:\Users\User\miniconda3\envs\schfs\Lib\site-packages\openpyxl\s

,relative_path,suffix,text_status,text_source,extracted_chars,text_preview
0,00_ASSET_MASTER\01_IDENTITY_REGISTERS\BOC0150-...,.zip,unsupported,unsupported,0,
1,00_ASSET_MASTER\01_IDENTITY_REGISTERS\ΓΕΜΗ_AET...,.pdf,ok,pdf,2979,ΕΛΛΗΝΙΚΗ ΔΗΜΟΚΡΑΤΙΑ ΕΜΠΟΡΙΚΟ & ΒΙΟΜΗΧΑΝΙΚΟ ΕΠΙ...
2,00_ASSET_MASTER\01_IDENTITY_REGISTERS\ΓΕΜΗ_AET...,.pdf,ok,pdf,7802,Σελίδα 1 από 4 ΚΑΤΑΣΤΑΤΙΚΟ ΣΥΣΤΑΣΗΣ ΤΗΣ ΙΔΙΩΤΙ...
3,00_ASSET_MASTER\01_IDENTITY_REGISTERS\ΓΕΜΗ_AET...,.pdf,ok,pdf,1370,ΕΛΛΗΝΙΚΗ ΔΗΜΟΚΡΑΤΙΑ ΕΜΠΟΡΙΚΟ & ΒΙΟΜΗΧΑΝΙΚΟ ΕΠΙ...
4,00_ASSET_MASTER\01_IDENTITY_REGISTERS\ΓΕΜΗ_AET...,.pdf,ok,pdf,1409,ΕΛΛΗΝΙΚΗ ΔΗΜΟΚΡΑΤΙΑ ΕΜΠΟΡΙΚΟ & ΒΙΟΜΗΧΑΝΙΚΟ ΕΠΙ...
5,00_ASSET_MASTER\02_MASTER_DATA\01 - Τι είναι τ...,.pdf,empty,pdf,0,
6,00_ASSET_MASTER\02_MASTER_DATA\01 - Τι είναι τ...,.docx,ok,docx,3215,Τι είναι ο βιοάνθρακας; Η Διεθνής Πρωτοβουλία ...
7,00_ASSET_MASTER\02_MASTER_DATA\01 - Τι είναι τ...,.pdf,ok,pdf,12015,A BRIEF UNDERSTANDING CARBON DIOXIDE REMOVAL J...
8,00_ASSET_MASTER\02_MASTER_DATA\01 - Τι είναι τ...,.pdf,ok,pdf,8343,European Biochar Market Report 2024 | 2025 4th...
9,00_ASSET_MASTER\02_MASTER_DATA\01 - Τι είναι τ...,.pdf,ok,pdf,4901,MAIN GOALS OF THE PROPOSAL Accelerate the depl...


In [6]:
display(enriched[enriched['text_status'] == 'error'][['relative_path', 'suffix', 'text_error']])

,relative_path,suffix,text_error


In [7]:
display(enriched['text_status'].value_counts(dropna=False).rename_axis('text_status').reset_index(name='count'))
display(enriched['text_source'].value_counts(dropna=False).rename_axis('text_source').reset_index(name='count'))
display(enriched[enriched['text_status'] == 'error'][['relative_path', 'suffix', 'text_error']].head(20))


,text_status,count
0,ok,158
1,empty,50
2,unsupported,41


,text_source,count
0,pdf,173
1,unsupported,41
2,xlsx,19
3,docx,16


,relative_path,suffix,text_error


In [8]:
display(enriched[enriched['has_extracted_text']][['relative_path', 'text_source', 'extracted_chars', 'text_preview']].head(20))


,relative_path,text_source,extracted_chars,text_preview
1,00_ASSET_MASTER\01_IDENTITY_REGISTERS\ΓΕΜΗ_AET...,pdf,2979,ΕΛΛΗΝΙΚΗ ΔΗΜΟΚΡΑΤΙΑ ΕΜΠΟΡΙΚΟ & ΒΙΟΜΗΧΑΝΙΚΟ ΕΠΙ...
2,00_ASSET_MASTER\01_IDENTITY_REGISTERS\ΓΕΜΗ_AET...,pdf,7802,Σελίδα 1 από 4 ΚΑΤΑΣΤΑΤΙΚΟ ΣΥΣΤΑΣΗΣ ΤΗΣ ΙΔΙΩΤΙ...
3,00_ASSET_MASTER\01_IDENTITY_REGISTERS\ΓΕΜΗ_AET...,pdf,1370,ΕΛΛΗΝΙΚΗ ΔΗΜΟΚΡΑΤΙΑ ΕΜΠΟΡΙΚΟ & ΒΙΟΜΗΧΑΝΙΚΟ ΕΠΙ...
4,00_ASSET_MASTER\01_IDENTITY_REGISTERS\ΓΕΜΗ_AET...,pdf,1409,ΕΛΛΗΝΙΚΗ ΔΗΜΟΚΡΑΤΙΑ ΕΜΠΟΡΙΚΟ & ΒΙΟΜΗΧΑΝΙΚΟ ΕΠΙ...
6,00_ASSET_MASTER\02_MASTER_DATA\01 - Τι είναι τ...,docx,3215,Τι είναι ο βιοάνθρακας; Η Διεθνής Πρωτοβουλία ...
7,00_ASSET_MASTER\02_MASTER_DATA\01 - Τι είναι τ...,pdf,12015,A BRIEF UNDERSTANDING CARBON DIOXIDE REMOVAL J...
8,00_ASSET_MASTER\02_MASTER_DATA\01 - Τι είναι τ...,pdf,8343,European Biochar Market Report 2024 | 2025 4th...
9,00_ASSET_MASTER\02_MASTER_DATA\01 - Τι είναι τ...,pdf,4901,MAIN GOALS OF THE PROPOSAL Accelerate the depl...
10,00_ASSET_MASTER\02_MASTER_DATA\01 - Τι είναι τ...,pdf,12015,"0 GLOBAL ESCO | 4, Chalkidi str., Moschato, At..."
12,00_ASSET_MASTER\02_MASTER_DATA\01 - Τι είναι τ...,pdf,12015,Lefebvre et al. Biochar (2023) 5:65 https://do...


In [9]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_base = OUTPUT_DIR / f'inventory_with_text_{timestamp}'
csv_path, parquet_path = save_text_outputs(enriched, output_base)
print('Saved CSV   :', csv_path)
print('Saved Parquet:', parquet_path)


Saved CSV   : c:\00_dev\SCH-FILE-ORGANIZER\data\outputs\inventory_with_text_20260330_083529.csv
Saved Parquet: c:\00_dev\SCH-FILE-ORGANIZER\data\outputs\inventory_with_text_20260330_083529.parquet


In [10]:
err = enriched[enriched['text_status'] == 'error'].copy()
long_err = err[err['path_length'] > 250].copy()
fnf_long = long_err[long_err['text_error'].fillna('').str.contains('FileNotFoundError', regex=False)]
print('errors_total =', len(err))
print('errors_path_gt_250 =', len(long_err))
print('FileNotFoundError_on_path_gt_250 =', len(fnf_long))
display(long_err[['relative_path', 'path_length', 'suffix', 'text_error']].head(20))

errors_total = 0
errors_path_gt_250 = 0
FileNotFoundError_on_path_gt_250 = 0


,relative_path,path_length,suffix,text_error


In [ ]:
import importlib
import src.executor
importlib.reload(src.executor)

from pathlib import Path
import os, shutil
import pandas as pd
from src.executor import copy_and_rename_from_paths

xlsx_path = Path(r"C:\SONADO-IK-801455240\Book2_refined_v17.xlsx")
if not xlsx_path.exists():
    raise FileNotFoundError(f"Missing Excel file: {xlsx_path}")

manifest_df = pd.read_excel(xlsx_path)
print(f"Loaded {len(manifest_df):,} rows from {xlsx_path.name}")

source_col = "absolute_path"
target_col = "refined_target_absolute_path_v4"

copy_manifest = manifest_df.copy()
copy_manifest["absolute_current_path"] = copy_manifest[source_col]
copy_manifest["absolute_proposed_path"] = copy_manifest[target_col]

log_df = copy_and_rename_from_paths(copy_manifest, overwrite_existing=True)
print(log_df[["copy_status", "copy_message"]].value_counts(dropna=False).rename("rows").to_string())

# --- move successful source files to _BACKUP ---
backup_root = Path(r"C:\SONADO-IK-801455240\_BACKUP")
source_root = Path(r"C:\SONADO-IK-801455240")

copied_rows = log_df[log_df["copy_status"] == "copied"].copy()
print(f"\nCopied rows eligible for backup transfer: {len(copied_rows)}")

backup_records = []
for src_raw in copied_rows["absolute_current_path"].dropna().astype(str):
    src = Path(src_raw.strip())
    rec = {
        "source": str(src),
        "backup_status": "pending",
        "backup_message": "",
        "backup_path": "",
    }
    try:
        if not src.exists():
            rec["backup_status"] = "missing_source"
            rec["backup_message"] = "source missing at backup step"
        elif not src.is_file():
            rec["backup_status"] = "skipped_non_file"
            rec["backup_message"] = "source exists but is not a file"
        else:
            try:
                rel = src.relative_to(source_root)
                dst = backup_root / rel
            except ValueError:
                # Fallback for paths outside source_root.
                dst = backup_root / "_UNMAPPED" / src.name

            dst.parent.mkdir(parents=True, exist_ok=True)

            # Avoid overwriting an existing backup file.
            if dst.exists():
                stem = dst.stem
                suffix = dst.suffix
                i = 1
                while True:
                    candidate = dst.with_name(f"{stem}__dup{i}{suffix}")
                    if not candidate.exists():
                        dst = candidate
                        break
                    i += 1

            shutil.move(str(src), str(dst))
            rec["backup_status"] = "moved_to_backup"
            rec["backup_message"] = "source moved to backup"
            rec["backup_path"] = str(dst)
    except Exception as exc:
        rec["backup_status"] = "error"
        rec["backup_message"] = f"backup move failed: {exc}"

    backup_records.append(rec)

backup_log_df = pd.DataFrame(backup_records)
if backup_log_df.empty:
    print("No files moved to backup.")
else:
    print("\nBackup transfer summary:")
    print(backup_log_df[["backup_status", "backup_message"]].value_counts(dropna=False).rename("rows").to_string())
    display(backup_log_df.head(20))

# --- delete empty folders under source_root (excluding _BACKUP) ---
removed_dirs = []
failed_dirs = []

# Bottom-up traversal allows removing nested empty folders first.
for root, dirs, files in os.walk(source_root, topdown=False):
    root_path = Path(root)

    # Never remove backup tree or source root itself.
    if root_path == source_root or backup_root in [root_path, *root_path.parents]:
        continue

    try:
        if not any(root_path.iterdir()):
            root_path.rmdir()
            removed_dirs.append(str(root_path))
    except Exception as exc:
        failed_dirs.append((str(root_path), str(exc)))

print(f"\nEmpty folder cleanup: removed={len(removed_dirs)} failed={len(failed_dirs)}")
if removed_dirs:
    print("Sample removed folders:")
    for p in removed_dirs[:20]:
        print(" -", p)
if failed_dirs:
    print("Sample failed removals:")
    for p, err in failed_dirs[:10]:
        print(f" - {p} | {err}")


Loaded 2,238 rows from Book2_refined_v17.xlsx
copy_status  copy_message             
copied       copy and rename completed    2238

missing_source count: 0


In [1]:
import pandas as pd

csv_path = r"c:\00_dev\SCH-FILE-ORGANIZER\data\outputs\inventory_with_text_20260309_154442.csv"
df = pd.read_csv(csv_path)

df.head()

,scan_root,absolute_path,relative_path,parent_relative,filename,stem,suffix,size_bytes,modified_at,created_at,...,text_status,text_source,extracted_text,text_preview,extracted_chars,text_truncated,text_error,extracted_pages,extracted_sheets,has_extracted_text
0,C:\SONADO-IK-801455240\HTL0049-01_OITYLO-KOKKA...,C:\SONADO-IK-801455240\HTL0049-01_OITYLO-KOKKA...,.DS_Store,NaN,.DS_Store,.DS_Store,NaN,14340,2026-02-27 09:59:43.850801229,2026-03-09 09:23:17.232372999,...,unsupported,unsupported,NaN,NaN,0,False,NaN,NaN,NaN,False
1,C:\SONADO-IK-801455240\HTL0049-01_OITYLO-KOKKA...,C:\SONADO-IK-801455240\HTL0049-01_OITYLO-KOKKA...,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\.DS_Store,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ,.DS_Store,.DS_Store,NaN,10244,2025-12-08 08:24:58.179627657,2026-03-09 09:23:18.215867043,...,unsupported,unsupported,NaN,NaN,0,False,NaN,NaN,NaN,False
2,C:\SONADO-IK-801455240\HTL0049-01_OITYLO-KOKKA...,C:\SONADO-IK-801455240\HTL0049-01_OITYLO-KOKKA...,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΔΗΛΩΣΕΙΣ ΑΝΑΘΕΣΗΣ - Α...,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΔΗΛΩΣΕΙΣ ΑΝΑΘΕΣΗΣ - Α...,01_SONADO_OITYLO_DHLWSH ANATHESHS.pdf,01_SONADO_OITYLO_DHLWSH ANATHESHS,.pdf,152816,2022-04-15 13:51:42.468893051,2026-03-09 09:23:18.831106901,...,ok,pdf,Δ Η Λ Ω Σ Η Α Ν Α Θ Ε Σ Ε Ω Ν\nΕΡΓΟ : ΑΝΕΓ...,Δ Η Λ Ω Σ Η Α Ν Α Θ Ε Σ Ε Ω Ν ΕΡΓΟ : ΑΝΕΓΕΡΣΗ ...,3412,False,NaN,1.0,NaN,True
3,C:\SONADO-IK-801455240\HTL0049-01_OITYLO-KOKKA...,C:\SONADO-IK-801455240\HTL0049-01_OITYLO-KOKKA...,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΔΗΛΩΣΕΙΣ ΑΝΑΘΕΣΗΣ - Α...,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΔΗΛΩΣΕΙΣ ΑΝΑΘΕΣΗΣ - Α...,02_SONADO_OITYLO_DHLWSH ANALHPSHS_STFNLAB OE.pdf,02_SONADO_OITYLO_DHLWSH ANALHPSHS_STFNLAB OE,.pdf,124890,2022-04-15 13:51:43.421580553,2026-03-09 09:23:19.272340298,...,ok,pdf,ΔΗΛΩΣΕΙΣ ΑΝΑΛΗΨΗΣ ΜΕΛΕΤΩΝ/ΕΠΙΒΛΕΨΕΩΝ\nΕΡΓΟ : ...,ΔΗΛΩΣΕΙΣ ΑΝΑΛΗΨΗΣ ΜΕΛΕΤΩΝ/ΕΠΙΒΛΕΨΕΩΝ ΕΡΓΟ : ΑΝ...,1206,False,NaN,1.0,NaN,True
4,C:\SONADO-IK-801455240\HTL0049-01_OITYLO-KOKKA...,C:\SONADO-IK-801455240\HTL0049-01_OITYLO-KOKKA...,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΔΗΛΩΣΕΙΣ ΑΝΑΘΕΣΗΣ - Α...,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΔΗΛΩΣΕΙΣ ΑΝΑΘΕΣΗΣ - Α...,03_SONADO_OITYLO_DHLWSH ANALHPSHS_GKA ENGINEER...,03_SONADO_OITYLO_DHLWSH ANALHPSHS_GKA ENGINEERS,.pdf,67290,2022-04-15 13:51:44.311928749,2026-03-09 09:23:19.710536003,...,empty,pdf,NaN,NaN,0,False,NaN,1.0,NaN,False


In [17]:
import re

# Excel/OpenXML disallows most control chars; keep tab/newline/carriage return.
illegal_ctrl = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F]")


def sanitize_for_excel(value):
    if isinstance(value, str):
        return illegal_ctrl.sub(" ", value)
    return value


df_export = df.copy()
text_cols = df_export.select_dtypes(include=["object", "string"]).columns
for col in text_cols:
    df_export[col] = df_export[col].map(sanitize_for_excel)

cols = [
    "scan_root", "absolute_path", "relative_path", "parent_relative",
    "filename", "stem", "suffix", "modified_at", "created_at",
    "depth_segments", "path_length", "hash", "is_duplicate_hash", "extracted_text",
    "text_preview", "has_extracted_text"
]

output_csv = "inventory_with_text_reduced_20260309_154442.csv"
df_export.to_csv(output_csv, columns=cols, index=False)
print(f"Saved: {output_csv}")

Saved: inventory_with_text_reduced_20260309_154442.csv


In [ ]:
from pathlib import Path
import hashlib
import os
import shutil
import pandas as pd


def apply_proposed_renames_from_excel(
    excel_path,
    root_path,
    source_col="source_relative_path",
    target_col="proposed_same_folder_path",
    duplicated_dir=r"C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASSETS\BCH01p800-01_vipe_SERRES\_DUPLICATED",
    sheet_name=0,
    dry_run=True,
    max_full_path_chars=240,
    max_filename_chars=120,
):
    """Rename files from source_relative_path -> proposed_same_folder_path.

    For duplicate target paths, keeps the newest source file (by mtime) as the winner
    and moves older source files to duplicated_dir.

    Overlong Windows targets are shortened deterministically before rename.
    """
    excel_path = Path(excel_path)
    root_path = Path(root_path)
    duplicated_dir = Path(duplicated_dir)

    df_map = pd.read_excel(excel_path, sheet_name=sheet_name)
    required = {source_col, target_col}
    missing = required - set(df_map.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    def norm_rel(value):
        return Path(str(value).strip().replace("/", "\\").lstrip("\\"))

    def to_os_path(path):
        raw = str(path)
        if os.name != "nt":
            return raw
        normalized = raw.replace("/", "\\")
        if normalized.startswith("\\\\?\\"):
            return normalized
        if normalized.startswith("\\\\"):
            return "\\\\?\\UNC\\" + normalized.lstrip("\\")
        return "\\\\?\\" + normalized

    def safe_exists(path: Path) -> bool:
        try:
            return os.path.exists(to_os_path(path))
        except OSError:
            return False

    def safe_mkdir(path: Path) -> None:
        os.makedirs(to_os_path(path), exist_ok=True)

    def unique_path(dst: Path) -> Path:
        if not safe_exists(dst):
            return dst
        stem, suffix = dst.stem, dst.suffix
        i = 1
        while True:
            candidate = dst.with_name(f"{stem}__dup{i}{suffix}")
            if not safe_exists(candidate):
                return candidate
            i += 1

    def shorten_target_path(dst: Path) -> tuple[Path, bool]:
        filename_budget = max_filename_chars
        if max_full_path_chars:
            parent_len = len(str(dst.parent))
            filename_budget = min(filename_budget, max(20, max_full_path_chars - parent_len - 1))

        if len(dst.name) <= filename_budget and (not max_full_path_chars or len(str(dst)) <= max_full_path_chars):
            return dst, False

        suffix = dst.suffix
        stem = dst.stem
        digest = hashlib.blake2b(str(dst).encode("utf-8"), digest_size=4).hexdigest()
        reserve = len(suffix) + len(digest) + 1
        stem_budget = max(8, filename_budget - reserve)
        short_stem = stem[:stem_budget].rstrip(" ._-") or stem[:stem_budget]
        short_name = f"{short_stem}_{digest}{suffix}"
        shortened = dst.with_name(short_name)

        if max_full_path_chars and len(str(shortened)) > max_full_path_chars:
            overflow = len(str(shortened)) - max_full_path_chars
            stem_budget = max(8, stem_budget - overflow)
            short_stem = stem[:stem_budget].rstrip(" ._-") or stem[:stem_budget]
            short_name = f"{short_stem}_{digest}{suffix}"
            shortened = dst.with_name(short_name)

        return shortened, True

    def safe_move(src: Path, dst: Path) -> None:
        safe_mkdir(dst.parent)
        shutil.move(to_os_path(src), to_os_path(dst))

    work = df_map[[source_col, target_col]].dropna().copy()
    work[source_col] = work[source_col].astype(str).str.strip()
    work[target_col] = work[target_col].astype(str).str.strip()
    work = work[(work[source_col] != "") & (work[target_col] != "")]
    work = work[work[source_col] != work[target_col]]

    ops = []
    for _, row in work.iterrows():
        src = root_path / norm_rel(row[source_col])
        raw_dst = root_path / norm_rel(row[target_col])
        final_dst, was_shortened = shorten_target_path(raw_dst)
        ops.append(
            {
                "source": src,
                "target": final_dst,
                "raw_target": raw_dst,
                "target_key": str(final_dst).lower(),
                "was_shortened": was_shortened,
            }
        )

    grouped = {}
    for rec in ops:
        grouped.setdefault(rec["target_key"], []).append(rec)

    winner_ops = []
    duplicate_losers = []

    for _, group in grouped.items():
        if len(group) == 1:
            winner_ops.append(group[0])
            continue

        existing = [r for r in group if safe_exists(r["source"])]
        if existing:
            winner = max(existing, key=lambda r: (Path(r["source"]).stat().st_mtime, str(r["source"]).lower()))
            winner_ops.append(winner)
            for r in group:
                if r is not winner:
                    duplicate_losers.append(r)
        else:
            duplicate_losers.extend(group)

    winner_src_set = {str(r["source"]).lower() for r in winner_ops}
    report_rows = []

    for rec in duplicate_losers:
        src = rec["source"]
        dst = rec["target"]

        if not safe_exists(src):
            report_rows.append(
                {
                    "source": str(src),
                    "target": str(dst),
                    "raw_target": str(rec["raw_target"]),
                    "status": "duplicate_source_missing",
                    "duplicate_action": "none",
                    "target_shortened": rec["was_shortened"],
                }
            )
            continue

        try:
            rel = src.relative_to(root_path)
            dup_target = duplicated_dir / rel
        except ValueError:
            dup_target = duplicated_dir / src.name

        dup_target, _ = shorten_target_path(dup_target)

        if dry_run:
            report_rows.append(
                {
                    "source": str(src),
                    "target": str(dst),
                    "raw_target": str(rec["raw_target"]),
                    "status": "would_move_to_duplicated",
                    "duplicate_action": str(dup_target),
                    "target_shortened": rec["was_shortened"],
                }
            )
            continue

        final_dup_target = unique_path(dup_target)
        safe_move(src, final_dup_target)
        report_rows.append(
            {
                "source": str(src),
                "target": str(dst),
                "raw_target": str(rec["raw_target"]),
                "status": "moved_to_duplicated",
                "duplicate_action": str(final_dup_target),
                "target_shortened": rec["was_shortened"],
            }
        )

    for rec in winner_ops:
        src = rec["source"]
        dst = rec["target"]

        if not safe_exists(src):
            report_rows.append(
                {
                    "source": str(src),
                    "target": str(dst),
                    "raw_target": str(rec["raw_target"]),
                    "status": "source_missing",
                    "duplicate_action": "winner_missing",
                    "target_shortened": rec["was_shortened"],
                }
            )
            continue

        target_exists_external = safe_exists(dst) and (str(dst).lower() not in winner_src_set)
        if target_exists_external:
            report_rows.append(
                {
                    "source": str(src),
                    "target": str(dst),
                    "raw_target": str(rec["raw_target"]),
                    "status": "blocked_target_exists",
                    "duplicate_action": "winner_blocked",
                    "target_shortened": rec["was_shortened"],
                }
            )
            continue

        status = "would_rename_shortened" if dry_run and rec["was_shortened"] else "would_rename"
        status = "renamed_shortened" if (not dry_run and rec["was_shortened"]) else status
        status = "renamed" if (not dry_run and not rec["was_shortened"]) else status

        if dry_run:
            report_rows.append(
                {
                    "source": str(src),
                    "target": str(dst),
                    "raw_target": str(rec["raw_target"]),
                    "status": status,
                    "duplicate_action": "winner",
                    "target_shortened": rec["was_shortened"],
                }
            )
            continue

        safe_move(src, dst)
        report_rows.append(
            {
                "source": str(src),
                "target": str(dst),
                "raw_target": str(rec["raw_target"]),
                "status": status,
                "duplicate_action": "winner",
                "target_shortened": rec["was_shortened"],
            }
        )

    report = pd.DataFrame(report_rows)
    if report.empty:
        summary = pd.DataFrame([{"status": "no_actions", "count": 0}])
    else:
        summary = (
            report.groupby("status", dropna=False)
            .size()
            .reset_index(name="count")
            .sort_values("count", ascending=False)
        )

    print(summary.to_string(index=False))
    return report, summary


# Example:
report, summary = apply_proposed_renames_from_excel(
    excel_path=r"C:\Users\User\Downloads\proposed_file_naming_20260330.xlsx",
    root_path=r"C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASSETS\BCH01p800-01_vipe_SERRES",
    duplicated_dir=r"C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASSETS\BCH01p800-01_vipe_SERRES\_DUPLICATED",
    dry_run=True,
)
report.head(20)


                  status  count
            would_rename    241
would_move_to_duplicated      5


,source,target,status,duplicate_action
0,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,would_move_to_duplicated,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...
1,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,would_move_to_duplicated,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...
2,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,would_move_to_duplicated,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...
3,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,would_move_to_duplicated,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...
4,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,would_move_to_duplicated,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...
5,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,would_rename,winner
6,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,would_rename,winner
7,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,would_rename,winner
8,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,would_rename,winner
9,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,would_rename,winner


In [25]:
report, summary = apply_proposed_renames_from_excel(
    excel_path=r"C:\Users\User\Downloads\proposed_file_naming_20260330.xlsx",
    root_path=r"C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASSETS\BCH01p800-01_vipe_SERRES",
    duplicated_dir=r"C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASSETS\BCH01p800-01_vipe_SERRES\_DUPLICATED",
    dry_run=False,
)
report.head(20)

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'C:\\Users\\User\\Desktop\\ATHBIO-IKE-803026649\\ASSETS\\BCH01p800-01_vipe_SERRES\\02_LAND_SITE\\01_OWNERSHIP_SURVEYS\\ΧΧΧΧΧΧΧΧΧΧΣΤΟΙΧΕΙΑ ΑΚΙΝΗΤΟΥ_ΒΙΠΕ_ΦΛΩΡΙΝΑΣ\\20. ΑΠΟΘΗΚΗ ΠΑΛΕΤΤΩΝ\\ΑΠΟΘΗΚΗ+ΠΑΛΕΤΩΝ+ΑΝΑΤΟΛΙΚΗ-ΔΥΤΙΚΗ+ΟΨΗ.pdf' -> 'C:\\Users\\User\\Desktop\\ATHBIO-IKE-803026649\\ASSETS\\BCH01p800-01_vipe_SERRES\\02_LAND_SITE\\01_OWNERSHIP_SURVEYS\\ΧΧΧΧΧΧΧΧΧΧΣΤΟΙΧΕΙΑ ΑΚΙΝΗΤΟΥ_ΒΙΠΕ_ΦΛΩΡΙΝΑΣ\\20. ΑΠΟΘΗΚΗ ΠΑΛΕΤΤΩΝ\\BCH01p800-01_LA_DRWTEC_pallet-warehouse-anatolike-dutike-elevation_20240618_v01_FINAL.pdf'

In [21]:
len('C:\\Users\\User\\Desktop\\ATHBIO-IKE-803026649\\ASSETS\\BCH01p800-01_vipe_SERRES\\02_LAND_SITE\\01_OWNERSHIP_SURVEYS\\ΧΧΧΧΧΧΧΧΧΧΣΤΟΙΧΕΙΑ ΑΚΙΝΗΤΟΥ_ΒΙΠΕ_ΦΛΩΡΙΝΑΣ\\20. ΑΠΟΘΗΚΗ ΠΑΛΕΤΤΩΝ\\BCH01p800-01_LA_DRWTEC_pallet-warehouse-anatolike-dutike-elevation_20240618_v01_FINAL.pdf')

261

In [15]:
from pathlib import Path

status_counts = report['status'].value_counts(dropna=False)
print('Status counts:')
print(status_counts.to_string())

check_statuses = ['source_missing', 'duplicate_source_missing']
subset = report[report['status'].isin(check_statuses)].copy()
print('\nRows with source_missing-like statuses:', len(subset))

samples = subset[['status', 'source', 'target']].head(30).copy()

def path_exists_variants(p):
    p = str(p)
    p1 = Path(p)
    p2 = Path(p.replace('\\', '/'))
    return p1.exists(), p2.exists()

samples['exists_as_is'] = samples['source'].map(lambda s: path_exists_variants(s)[0])
samples['exists_swapped_slash'] = samples['source'].map(lambda s: path_exists_variants(s)[1])
print('\nSample missing rows with slash check:')
print(samples.to_string(index=False))

print('\nAny row where swapped slashes fixes existence?')
print(((samples['exists_as_is'] == False) & (samples['exists_swapped_slash'] == True)).any())

Status counts:
status
source_missing              237
duplicate_source_missing      9

Rows with source_missing-like statuses: 246

Sample missing rows with slash check:
                  status                                                                                                                                                                                            source                                                                                                                                                                                                                         target  exists_as_is  exists_swapped_slash
duplicate_source_missing                                    C:\Users\User\Desktop\ATHBIO-IKE-803026649\03_PERMITTING_APPROVALS\01_PERMITS_LICENSES\04 - ΑΕΠΟ\ΑΕΠΟ_30.10.25\ΜΠΕ_ΜΟΝΑΔΑ ΠΑΡΑΓΩΓΗΣ BIOCHAR_final_s_signed.pdf                      C:\Users\User\Desktop\ATHBIO-IKE-803026649\03_PERMITTING_APPROVALS\01_PERMITS_LICENSES\04 - ΑΕΠΟ\ΑΕΠΟ_30.10.25\B

In [16]:
subset = report[report['status'].isin(['source_missing', 'duplicate_source_missing'])].copy()
slash_fix_count = 0
for s in subset['source'].head(200):
    p1 = Path(str(s))
    p2 = Path(str(s).replace('\\', '/'))
    if (not p1.exists()) and p2.exists():
        slash_fix_count += 1
print({'missing_like_rows': int(len(subset)), 'slash_fix_count_in_sample200': int(slash_fix_count)})

{'missing_like_rows': 246, 'slash_fix_count_in_sample200': 0}


In [23]:
print(report['status'].value_counts(dropna=False).to_string())

miss = report[report['status'].isin(['source_missing','duplicate_source_missing'])].copy()
print('\nmissing rows:', len(miss))
if not miss.empty:
    print('\nSample sources:')
    print(miss['source'].head(5).to_string(index=False))
    print('\nCommon path prefixes:')
    print(miss['source'].astype(str).str.extract(r'^(.{0,120})')[0].head(5).to_string(index=False))

status
source_missing              237
duplicate_source_missing      9

missing rows: 246

Sample sources:
C:\Users\User\Desktop\ATHBIO-IKE-803026649\03_P...
C:\Users\User\Desktop\ATHBIO-IKE-803026649\03_P...
C:\Users\User\Desktop\ATHBIO-IKE-803026649\03_P...
C:\Users\User\Desktop\ATHBIO-IKE-803026649\03_P...
C:\Users\User\Desktop\ATHBIO-IKE-803026649\05_C...

Common path prefixes:
C:\Users\User\Desktop\ATHBIO-IKE-803026649\03_P...
C:\Users\User\Desktop\ATHBIO-IKE-803026649\03_P...
C:\Users\User\Desktop\ATHBIO-IKE-803026649\03_P...
C:\Users\User\Desktop\ATHBIO-IKE-803026649\03_P...
C:\Users\User\Desktop\ATHBIO-IKE-803026649\05_C...
